In [168]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np

In [137]:
import pandas as pd
import pytz
from timezonefinder import TimezoneFinder
from datetime import datetime

# open hourly ERA5

In [189]:
ds = xr.open_dataset('/Users/gahe5417/data/ERA5/hourly_air.2m.era5.CONUS.January.2001.nc')
ds = ds.sel(time=slice('2001-01-01 08:00:00', '2001-01-31 15:59:59'))

# Define arrays

In [190]:
local_time = xr.zeros_like(ds['air'])
time_zone = xr.zeros_like(ds['air'].mean('time')).astype('str')
tavg = xr.zeros_like(ds['air'])
tmax = xr.zeros_like(ds['air'].resample(time='1D').mean()).rename({'time':'local_time'})
tmin = xr.zeros_like(ds['air'].resample(time='1D').mean()).rename({'time':'local_time'})

# Define local time

In [191]:
# 1. Initialize the finder
tf = TimezoneFinder()
time_index = ds["time"].to_index()

In [192]:
for j in range(len(ds["lat"])):
    for i in range(len(ds["lon"])):
      # 2. Provide your coordinates (e.g., Times Square, NY)
      lat, lon = ds["lat"].values[j], ds["lon"].values[i]

      # 3. Get the Timezone ID (e.g., 'America/New_York')
      tz_string = tf.timezone_at(lng=lon-360, lat=lat)

      # 4. Use pytz to create a timezone-aware datetime object
      local_tz = pytz.timezone(tz_string)
      time_zone.loc[{'lat':lat,'lon':lon}] = tz_string
      local_time.loc[{'lat':lat,'lon':lon}] = time_index.tz_localize("UTC").tz_convert(local_tz).to_pydatetime().astype('datetime64[ns]')

/var/folders/41/jpdddhvx1_bfhbywzg083cy00000gt/T/ipykernel_41058/59790745.py:12: UserWarning: no explicit representation of timezones available for np.datetime64
  local_time.loc[{'lat':lat,'lon':lon}] = time_index.tz_localize("UTC").tz_convert(local_tz).to_pydatetime().astype('datetime64[ns]')


# Assign new time as coordinate

In [193]:
local_time = local_time.rename('local_time').astype('datetime64[ns]') 
ds = ds.assign_coords(local_time=local_time)
ds = ds.assign_coords(time_zone=time_zone)

# Compute Tmin and Tmax by grouping on local time

In [194]:
np.unique(time_zone.values)

array(['America/Atikokan', 'America/Boise', 'America/Chicago',
       'America/Chihuahua', 'America/Ciudad_Juarez', 'America/Creston',
       'America/Denver', 'America/Detroit', 'America/Edmonton',
       'America/Hermosillo', 'America/Indiana/Indianapolis',
       'America/Indiana/Knox', 'America/Indiana/Marengo',
       'America/Indiana/Petersburg', 'America/Indiana/Tell_City',
       'America/Indiana/Vincennes', 'America/Indiana/Winamac',
       'America/Kentucky/Louisville', 'America/Kentucky/Monticello',
       'America/Los_Angeles', 'America/Matamoros', 'America/Mazatlan',
       'America/Menominee', 'America/Mexico_City', 'America/Moncton',
       'America/Monterrey', 'America/Nassau', 'America/New_York',
       'America/North_Dakota/Beulah', 'America/North_Dakota/Center',
       'America/North_Dakota/New_Salem', 'America/Ojinaga',
       'America/Phoenix', 'America/Regina', 'America/Swift_Current',
       'America/Tijuana', 'America/Toronto', 'America/Vancouver',
       'Ameri

In [195]:
for tz in np.unique(time_zone.values):
  idy,idx = np.where(time_zone == tz)
  ilat = ds['lat'].isel(lat=idy)
  ilon = ds['lon'].isel(lon=idx)
  local_day = ds.local_time.sel(lon=ilon[0],lat=ilat[0]).dt.day
  tmin[:,idy,idx] = ds['air'].sel(lon=ilon,lat=ilat).groupby(local_day).min("time").data
  tmax[:,idy,idx] = ds['air'].sel(lon=ilon,lat=ilat).groupby(local_day).max("time").data